In [0]:
df = spark.read.format("delta").load(
    "/Volumes/workspace/default/raw_datasets/feature_engineering"
    
)

In [0]:
import gc
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
 
print("="*48)
print("  PHASE 3: FEATURE ENGINEERING")
print("         (Serverless-Compatible Build)")
print("="*48)

In [0]:
 
df_model = df.drop("customerID", "Churn")
print(f"Columns after drop: {len(df_model.columns)}")
print("Columns:", df_model.columns)
 

In [0]:
categorical_cols = [
    "gender", "Partner", "Dependents", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling",
    "PaymentMethod"
]
 
numerical_cols = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]
 
print(f"\nCategorical features : {len(categorical_cols)}")
print(f"Numerical features   : {len(numerical_cols)}")
print(f"Target column        : label")
 

In [0]:
print("\nBuilding pipeline stages...")
 
all_stages = (
    # Stage group 1 — StringIndexers (14 columns)
    [
        StringIndexer(
            inputCol=c,
            outputCol=c + "_idx",
            handleInvalid="keep"
        )
        for c in categorical_cols
    ]
    +
    # Stage group 2 — OneHotEncoders (14 columns)
    [
        OneHotEncoder(
            inputCols =[c + "_idx" for c in categorical_cols],
            outputCols=[c + "_enc" for c in categorical_cols],
            dropLast=True
        )
    ]   # ← ONE multi-column encoder instead of 14 separate ones
        #   This alone cuts cache objects from 14 down to 1
    +
    # Stage group 3 — Numeric assembler + scaler
    [
        VectorAssembler(
            inputCols=numerical_cols,
            outputCol="numerical_raw",
            handleInvalid="keep"
        ),
        StandardScaler(
            inputCol="numerical_raw",
            outputCol="numerical_scaled",
            withMean=True,
            withStd=True
        ),
    ]
    +
    # Stage group 4 — Final assembler
    [
        VectorAssembler(
            inputCols=[c + "_enc" for c in categorical_cols] + ["numerical_scaled"],
            outputCol="features",
            handleInvalid="keep"
        )
    ]
)
 
pipeline = Pipeline(stages=all_stages)
 
# Free the stage list — no longer needed after Pipeline holds them
del all_stages
gc.collect()
 
print(f"Pipeline built: {len(pipeline.getStages())} stages")
print("  StringIndexers  : 14")
print("  OneHotEncoder   :  1  (multi-column — saves cache space)")
print("  NumAssembler    :  1")
print("  StandardScaler  :  1")
print("  FinalAssembler  :  1")
 

In [0]:
print("\nFitting pipeline (this may take ~1-2 min on free tier)...")
pipeline_model = pipeline.fit(df_model)
 
# Free the unfitted pipeline object right away
del pipeline
gc.collect()
 
print("Transforming data...")
df_prepared = pipeline_model.transform(df_model)
 
# Keep only the two columns needed downstream — drop all
# intermediate encoding columns to cut memory footprint
df_prepared = df_prepared.select("features", "label")
 
# Free df_model — no longer needed
del df_model
gc.collect()
 
print("Pipeline complete.")
 
# Verify feature vector (one collect call only)
sample_vec = df_prepared.select("features").first()[0]
print(f"Feature vector size : {len(sample_vec)} dimensions")
 

In [0]:
print("\nSample — features vector and label:")
display(df_prepared.select("features", "label").limit(3))
 
 
# ── CELL 7: Train / Test split ────────────────────────────
train_df, test_df = df_prepared.randomSplit([0.8, 0.2], seed=42)
 
# Free df_prepared now that we have train/test splits
del df_prepared
gc.collect()
 
print("Train/test split done (80/20, seed=42)")

In [0]:
TRAIN_PATH = "/Volumes/workspace/default/raw_datasets/train_delta"
TEST_PATH  = "/Volumes/workspace/default/raw_datasets/test_delta"
 
print("Writing train set to Delta checkpoint...")
train_df.write.format("delta").mode("overwrite").save(TRAIN_PATH)
 
print("Writing test set to Delta checkpoint...")
test_df.write.format("delta").mode("overwrite").save(TEST_PATH)
 
# Free the lazy DataFrames — Delta files are the source of truth now
del train_df, test_df
gc.collect()
 
# Read back as Delta — fast columnar reads, no cache needed
train_df = spark.read.format("delta").load(TRAIN_PATH)
test_df  = spark.read.format("delta").load(TEST_PATH)
 
train_cnt = train_df.count()
test_cnt  = test_df.count()
total     = train_cnt + test_cnt
 
print(f"\nTrain set : {train_cnt:,} rows ({train_cnt/total*100:.0f}%)")
print(f"Test set  : {test_cnt:,}  rows ({test_cnt/total*100:.0f}%)")
 
print("\nChurn distribution in train set:")
display(train_df.groupBy("label").count().orderBy("label"))
 
print("Churn distribution in test set:")
display(test_df.groupBy("label").count().orderBy("label"))
 

In [0]:
PIPELINE_PATH = "/Volumes/workspace/default/raw_datasets/churn_pipeline"
pipeline_model.write().overwrite().save(PIPELINE_PATH)
print(f"\nPipeline model saved to: {PIPELINE_PATH}")
 
# Free pipeline_model from cache after saving
del pipeline_model
gc.collect()
 
print("\nReload later with:")
print("  from pyspark.ml import PipelineModel")
print(f"  pipeline_model = PipelineModel.load('{PIPELINE_PATH}')")

In [0]:
print("\n" + "="*48)
print("  SESSION STATE AFTER PHASE 3")
print("="*48)
print("  train_df  — Delta checkpoint ✓  (ready for model training)")
print("  test_df   — Delta checkpoint ✓  (ready for evaluation)")
print("  df        — still available (original cleaned data)")
print(f"\n  Train Delta : {TRAIN_PATH}")
print(f"  Test Delta  : {TEST_PATH}")
print("\n PHASE 3 COMPLETE — proceed to Phase 4: Model Training")